In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))



/kaggle/input/datasets/mragpavank/diabetes/diabetes.csv


In [2]:
df = pd.read_csv('/kaggle/input/datasets/mragpavank/diabetes/diabetes.csv')

In [3]:
df.sample(4)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
115,4,146,92,0,0,31.2,0.539,61,1
320,4,129,60,12,231,27.5,0.527,31,0
533,6,91,0,0,0,29.8,0.501,31,0
654,1,106,70,28,135,34.2,0.142,22,0


In [4]:
df.shape

(768, 9)

In [5]:
X = df.iloc[:,:-1].values
y = df.iloc[:,-1].values

In [6]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [7]:
X = scaler.fit_transform(X)

In [8]:
from sklearn.model_selection import train_test_split

In [9]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state = 42)

In [10]:
import tensorflow
from tensorflow import keras
from keras import Sequential 
from keras.layers import Dense

In [11]:
model = Sequential()
model.add(Dense(32,activation = 'relu',input_dim=8))
model.add(Dense(1,activation = 'sigmoid'))

model.compile(optimizer = 'Adam',loss='binary_crossentropy',metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-09-19 11:28:59.909824: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [12]:
model.fit(X_train,y_train,batch_size=32,epochs=23,validation_data=(X_test,y_test))

Epoch 1/23
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6726 - loss: 0.6520 - val_accuracy: 0.6623 - val_loss: 0.6301
Epoch 2/23
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6971 - loss: 0.5900 - val_accuracy: 0.6948 - val_loss: 0.5844
Epoch 3/23
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7199 - loss: 0.5504 - val_accuracy: 0.7208 - val_loss: 0.5567
Epoch 4/23
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7362 - loss: 0.5245 - val_accuracy: 0.7143 - val_loss: 0.5397
Epoch 5/23
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7573 - loss: 0.5071 - val_accuracy: 0.7273 - val_loss: 0.5274
Epoch 6/23
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7638 - loss: 0.4940 - val_accuracy: 0.7403 - val_loss: 0.5185
Epoch 7/23
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7687 - loss: 0.4850 - val_accuracy: 0.7468 - val_loss: 0.5138
Epoch 8/23
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7736 - loss: 0.4766 - val_accuracy: 0.7468 - val_loss

In [13]:
import keras_tuner as kt

In [30]:
def build_model(hp):
    
    model = Sequential()
    
    model.add(Dense(32, activation='relu', input_dim=8))
    model.add(Dense(1, activation='sigmoid'))
    
    optimizer = hp.Choice(
        'optimizer',
        values=['adam', 'sgd', 'adagrad', 'rmsprop']
    )
    
    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

In [31]:
tuner = kt.RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=5,
)

In [32]:
tuner.search(
    X_train,
    y_train,
    epochs=5,
    validation_data=(X_test, y_test)
)

Trial 4 Complete [00h 00m 02s]
val_accuracy: 0.7727272510528564

Best val_accuracy So Far: 0.7727272510528564
Total elapsed time: 00h 00m 09s


In [34]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'adam'}

In [35]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [40]:
model.fit(X_train,y_train,epochs=28,batch_size=32,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/28
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7932 - loss: 0.4316 - val_accuracy: 0.7662 - val_loss: 0.5213
Epoch 8/28
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7948 - loss: 0.4310 - val_accuracy: 0.7662 - val_loss: 0.5203
Epoch 9/28
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7980 - loss: 0.4298 - val_accuracy: 0.7727 - val_loss: 0.5204
Epoch 10/28
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7964 - loss: 0.4285 - val_accuracy: 0.7727 - val_loss: 0.5229
Epoch 11/28
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7997 - loss: 0.4278 - val_accuracy: 0.7727 - val_loss: 0.5233
Epoch 12/28
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8013 - loss: 0.4270 - val_accuracy: 0.7727 - val_loss: 0.5217
Epoch 13/28
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7997 - loss: 0.4260 - val_accuracy: 0.7727 - val_loss: 0.5254
Epoch 14/28
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8013 - loss: 0.4252 - val_accuracy: 0.7727 - val

In [52]:
def build_model(hp):
    model = Sequential()
    counter = 0
    for i in range(hp.Int('num_layers',min_value=1,max_value=10)):
        if counter ==0:
            model.add(
                Dense(
                hp.Int('units' + str(i),min_value=8,max_value=128,step=8),
                activation= hp.Choice('activation',values=['relu','tanh','sigmoid']), 
                input_dim=8))
        else:
            model.add(
                Dense(
                hp.Int('units' + str(i),min_value=8,max_value=128,step=8),
                activation= hp.Choice('activation',values=['relu','tanh','sigmoid'])))
        counter+=1
    model.add(Dense(1, activation='sigmoid'))
    
    optimizer = hp.Choice(
        'optimizer',
        values=['adam', 'sgd', 'adagrad', 'rmsprop']
    )
    
    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

In [53]:
tuner = kt.RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=5,
    directory='my_dir',
    project_name='my_project'
)

Reloading Tuner from my_dir/my_project/tuner0.json


In [54]:
tuner.search(
    X_train,
    y_train,
    epochs=10,
    validation_data=(X_test, y_test)
)

Trial 5 Complete [00h 00m 03s]
val_accuracy: 0.7467532753944397

Best val_accuracy So Far: 0.7467532753944397
Total elapsed time: 00h 00m 44s


In [56]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 10,
 'units0': 40,
 'activation': 'tanh',
 'optimizer': 'sgd',
 'units1': 120,
 'units2': 72,
 'units3': 8,
 'units4': 40,
 'units5': 104,
 'units6': 8,
 'units7': 120,
 'units8': 8,
 'units9': 8}

In [57]:
model = tuner.get_best_models(num_models=1)[0]

In [58]:
model.fit(X_train,y_train,epochs=40,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/40
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.7720 - loss: 0.4918 - val_accuracy: 0.7403 - val_loss: 0.5194
Epoch 8/40
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7687 - loss: 0.4863 - val_accuracy: 0.7468 - val_loss: 0.5172
Epoch 9/40
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7704 - loss: 0.4820 - val_accuracy: 0.7403 - val_loss: 0.5170
Epoch 10/40
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7606 - loss: 0.4796 - val_accuracy: 0.7532 - val_loss: 0.5113
Epoch 11/40
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7752 - loss: 0.4766 - val_accuracy: 0.7403 - val_loss: 0.5089
Epoch 12/40
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7704 - loss: 0.4756 - val_accuracy: 0.7468 - val_loss: 0.5108
Epoch 13/40
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7752 - loss: 0.4740 - val_accuracy: 0.7403 - val_loss: 0.5151
Epoch 14/40
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7785 - loss: 0.4737 - val_accuracy: 0.7532 - val

In [64]:
from keras.layers import Dropout

In [65]:
def build_model(hp):
    
    model = Sequential()
    counter = 0
    
    for i in range(hp.Int('num_layers', min_value=1, max_value=10)):
        
        if counter == 0:
            model.add(
                Dense(
                    hp.Int(
                        'units' + str(i),
                        min_value=8,
                        max_value=128,
                        step=8
                    ),
                    activation=hp.Choice(
                        'activation',
                        values=['relu', 'tanh', 'sigmoid']
                    ),
                    input_dim=8
                )
            )
        else:
            model.add(
                Dense(
                    hp.Int(
                        'units' + str(i),
                        min_value=8,
                        max_value=128,
                        step=8
                    ),
                    activation=hp.Choice(
                        'activation',
                        values=['relu', 'tanh', 'sigmoid']
                    )
                )
            )
        
        # Add dropout after every Dense layer
        model.add(
            Dropout(
                hp.Float(
                    'dropout_' + str(i),
                    min_value=0.0,
                    max_value=0.5,
                    step=0.1
                )
            )
        )
        
        counter += 1
    
    model.add(Dense(1, activation='sigmoid'))
    
    optimizer = hp.Choice(
        'optimizer',
        values=['adam', 'sgd', 'adagrad', 'rmsprop']
    )
    
    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

In [60]:
tuner = kt.RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=5,
    directory='my_dir',
    project_name='my_project'
)

Reloading Tuner from my_dir/my_project/tuner0.json


In [66]:
tuner.search(
    X_train,
    y_train,
    epochs=10,
    validation_data=(X_test, y_test)
)

In [67]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 10,
 'units0': 40,
 'activation': 'tanh',
 'optimizer': 'sgd',
 'units1': 120,
 'units2': 72,
 'units3': 8,
 'units4': 40,
 'units5': 104,
 'units6': 8,
 'units7': 120,
 'units8': 8,
 'units9': 8}

In [68]:
model = tuner.get_best_models(num_models=1)[0]

In [69]:
model.fit(X_train,y_train,epochs=40,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/40
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.7752 - loss: 0.4909 - val_accuracy: 0.7468 - val_loss: 0.5211
Epoch 8/40
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7622 - loss: 0.4874 - val_accuracy: 0.7208 - val_loss: 0.5174
Epoch 9/40
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7752 - loss: 0.4838 - val_accuracy: 0.7338 - val_loss: 0.5169
Epoch 10/40
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7704 - loss: 0.4808 - val_accuracy: 0.7338 - val_loss: 0.5152
Epoch 11/40
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7655 - loss: 0.4774 - val_accuracy: 0.7078 - val_loss: 0.5240
Epoch 12/40
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7736 - loss: 0.4755 - val_accuracy: 0.7403 - val_loss: 0.5096
Epoch 13/40
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7785 - loss: 0.4747 - val_accuracy: 0.7597 - val_loss: 0.4989
Epoch 14/40
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7687 - loss: 0.4737 - val_accuracy: 0.7597 - val